# LLM Function Call 示例

本 Notebook 演示 **大模型工具调用（Tool / Function Calling）** 的完整流程：用户提问 → 模型决定调用哪个函数 → 本地执行函数 → 模型根据结果生成最终回答。

## 核心组件

| 组件 | 说明 |
|------|------|
| `tools` | 向模型注册的函数 Schema（名称、描述、参数） |
| `get_weather()` | 本地实际执行的 Mock 天气查询函数 |
| `send_messages()` | 封装 API 调用，传入 `messages_history` + `tools` |
| `messages_history` | 多轮对话历史，贯穿整个调用链 |

## 代码逻辑图

```mermaid
flowchart LR
    A[用户提问] --> B[LLM<br/>返回 tool_call]
    B --> C[执行<br/>get_weather]
    C --> D[LLM<br/>生成回答]
```


In [1]:
from openai import OpenAI
import json
from pprint import pprint

client = OpenAI(
    api_key="sk-your-api-key-here",
    base_url="https://api.deepseek.com",
)

def to_api_message(msg):
    """统一转为 API 可接受的 dict 格式。"""
    if hasattr(msg, "model_dump"):
        return msg.model_dump(exclude_none=True)
    return msg

def send_messages(messages_history, tools):
    response = client.chat.completions.create(
        model="deepseek-v4-pro",
        messages=[to_api_message(m) for m in messages_history],
        tools=tools
    )
    return response.choices[0].message

def print_message(message, style="json"):
    """格式化打印单条消息（dict 或 ChatCompletionMessage），自动去掉 None 字段。"""
    if hasattr(message, "model_dump"):
        data = message.model_dump(exclude_none=True)
    else:
        data = {k: v for k, v in message.items() if v is not None}
    if style == "pprint":
        pprint(data)
    else:
        print(json.dumps(data, indent=2, ensure_ascii=False))

def print_messages(messages_history, style="json"):
    for i, msg in enumerate(messages_history):
        role = msg.get("role") if isinstance(msg, dict) else getattr(msg, "role", "assistant")
        print(f"--- [{i}] {role} ---")
        print_message(msg, style)


In [2]:
def get_weather(location: str) -> str:
    """Mock weather lookup — replace with a real API call if needed."""
    mock_data = {
        "hangzhou": "24℃，晴，东南风 3 级",
        "beijing": "18℃，多云，北风 2 级",
        "shanghai": "26℃，阴，东风 2 级",
    }
    key = location.lower().split(",")[0].strip()
    return mock_data.get(key, f"{location} 的天气暂无数据")

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of a location, the user should supply a location first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"]
            },
        }
    },
]    

In [3]:
messages_history = [{"role": "user", "content": "How's the weather in Hangzhou, Zhejiang?"}]
message = send_messages(messages_history, tools)
print(f"User>\t {messages_history[0]['content']}")

User>	 How's the weather in Hangzhou, Zhejiang?


In [4]:
print_message(message)

{
  "content": "",
  "role": "assistant",
  "tool_calls": [
    {
      "id": "call_00_AVX5pYDa8P7Ret7iqXj58243",
      "function": {
        "arguments": "{\"location\": \"Hangzhou, Zhejiang\"}",
        "name": "get_weather"
      },
      "type": "function",
      "index": 0
    }
  ],
  "reasoning_content": "The user is asking about the weather in Hangzhou, Zhejiang. I need to call the get_weather function with the location \"Hangzhou, Zhejiang\"."
}


In [5]:
tool = message.tool_calls[0]
# 必须先追加 assistant（含 tool_calls），再追加 tool 结果
messages_history.append(message.model_dump(exclude_none=True))

In [6]:
# 解析 LLM 返回的函数调用参数，执行真正的 get_weather
tool_args = json.loads(tool.function.arguments)
weather_result = get_weather(tool_args["location"])
messages_history.append({"role": "tool", "tool_call_id": tool.id, "content": weather_result})

In [7]:
print(weather_result)

24℃，晴，东南风 3 级


In [33]:
print_messages(messages_history)

--- [0] user ---
{
  "role": "user",
  "content": "How's the weather in Hangzhou, Zhejiang?"
}
--- [1] assistant ---
{
  "content": "",
  "role": "assistant",
  "tool_calls": [
    {
      "id": "call_00_XBhWuSRPh729sioyTA8u3189",
      "function": {
        "arguments": "{\"location\": \"Hangzhou, Zhejiang\"}",
        "name": "get_weather"
      },
      "type": "function",
      "index": 0
    }
  ],
  "reasoning_content": "The user is asking about the weather in Hangzhou, Zhejiang. I need to call the get_weather function with the appropriate location parameter."
}
--- [2] tool ---
{
  "role": "tool",
  "tool_call_id": "call_00_XBhWuSRPh729sioyTA8u3189",
  "content": "24℃，晴，东南风 3 级"
}


In [8]:
message = send_messages(messages_history, tools)
print(f"Model>\t {message.content}")

Model>	 杭州现在的天气不错！☀️

- 🌡️ **气温**：24℃
- ☀️ **天气**：晴
- 🌬️ **风力**：东南风 3 级

整体来说，是非常舒适宜人的天气，适合出门活动。


In [9]:
print_message(message)

{
  "content": "杭州现在的天气不错！☀️\n\n- 🌡️ **气温**：24℃\n- ☀️ **天气**：晴\n- 🌬️ **风力**：东南风 3 级\n\n整体来说，是非常舒适宜人的天气，适合出门活动。",
  "role": "assistant",
  "reasoning_content": "The weather result for Hangzhou, Zhejiang is: 24°C, sunny (晴), southeast wind level 3 (东南风 3 级). I'll present this nicely to the user."
}
